# 00 — API Data Gathering

Fetch PubMed articles via the E-utilities API and save as **monthly JSONL** files.

**Query scope:** All US-affiliated, English-language journal articles with abstracts (humans only).
**No disease-term pre-filtering** — disease detection is handled downstream (NER + MeSH, notebook 03).
**Period:** Jan 1994 – Dec 2025. Estimated total: 3–5M articles.

## Design

1. **Date-range splitting to beat the 10k wall.** For `db=pubmed` no result set can be
   paged past offset 10,000 (esearch caps at 10k PMIDs; efetch-from-history returns 400
   for `retstart>=10000`). The history server does not lift this. Each month is split
   into date-range sub-queries (month → days → half-days) until every piece returns
   under `SAFE_SET_MAX` (9,500) and can be paged in full.

2. **Piece-based resume with PMID dedup.** `progress.json` tracks `completed_months`
   and an `in_progress` block listing finished `done_pieces`. Resume skips finished
   pieces and re-runs the rest, deduping by PMID. Safe because historical months are
   frozen sets.

3. **Completeness by summing pieces.** A month completes once all its date-range
   pieces finish. The gap `month_total − unique_saved` (deleted or withheld PMIDs,
   which is normal) is logged but never fatal.

4. **JSONL append.** O(1) append, `fsync`'d, with torn-line repair on resume. Replaces
   the previous O(n²) read-concat-rewrite and its corruption window.

5. **Robust HTTP.** Timeouts on every call; retries on `RequestException`,
   `ET.ParseError`, HTTP 429 and 5xx; exponential backoff with jitter; one global
   rate limiter with headroom under the NCBI ceiling.

6. **NCBI identification.** Every call carries `tool` and `email` params plus a
   User-Agent header, per NCBI policy.

7. **Full-record fetch** via `retmode=xml` so MeSH qualifiers and `CoiStatement` are
   always present.

8. **Faithful XML extraction.** `itertext()` walks all text nodes (no truncation at
   inline `<i>` or `<sup>`). `MedlineDate` ("1997 Jan-Feb", "1998 Spring") and
   `ArticleDate` fallbacks are handled. Per-month parse-failure count is logged.

9. **Single efetch POST of 200 PMIDs per call**, not 20 calls of 10.

## Date precision

Each record carries `pubdate_raw` and `pubdate_precision`
(`full_date` / `year_month` / `year`), so "1998 Spring" (year precision) is not
mistaken for January in seasonality analysis. PubMed's `[PDAT]` buckets by
publication date — not the date added to PubMed, which is `[EDAT]` / `[CRDT]` —
so bucketing is correct; the flag captures the residual granularity loss.

## Failure handling

- **Per-chunk parse retry with a missing-PMID manifest.** A transient bad efetch
  chunk is retried, not fatal to the month. PMIDs that efetch never returns
  (deleted or withheld — normal) are logged to `errors/missing_*.json` and the
  month still completes, so it cannot get stuck forever.
- **Adaptive throttle.** Repeated 429s widen the request interval. This also
  covers a silently-invalid API key, which drops the limit to the 3 req/s IP cap.
- **Forced real email** (placeholder addresses rejected at startup).
- **File + console logging** for the unattended run.
- **Newline-safe append** so a torn line cannot fuse with the next record.
- **Empty file** written for months with zero articles (so resume can distinguish
  "done, no results" from "never ran").
- **Within-batch dedup** before write.
- **Error-file cleanup** on month success.

## 0. Libraries

In [ ]:
import os
import re
import sys
import json
import glob
import time
import random
import logging
import calendar
import tempfile
from getpass import getpass
from typing import Optional
from xml.etree import ElementTree as ET

import requests

## 1. Configuration
Set your API key, **contact email**, and paths here. Run the notebook from repo root.

In [ ]:
# Load NCBI credentials from .env so secrets stay out of the notebook.
# find_dotenv() walks up from cwd to locate the file, so this works
# whether the notebook runs from notebooks/ or from the project root.

from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())

API_KEY = os.getenv("NCBI_API_KEY")   
EMAIL   = os.getenv("EMAIL")          

# Sanity check — presence only, never the value.
print("API_KEY loaded:", API_KEY is not None)
print("EMAIL loaded:  ", EMAIL is not None)

In [ ]:
# ── Identity ────────────────────────────────────────────────────────────────
# NCBI asks every caller to identify itself so they can warn before throttling.
# API_KEY and EMAIL are loaded from .env in the previous cell.
TOOL = "biomedical-literature-analysis"

if "@" not in EMAIL or EMAIL.split("@")[-1].lower() in {"example.com", "example.org", "email.com"}:
    raise ValueError("Set EMAIL to a real contact address before running (NCBI requires it).")

# ── Project root ────────────────────────────────────────────────────────────
# Resolve from the notebook's location, not the cwd, so the same notebook works
# whether Jupyter starts in notebooks/ or at the project root.
# Layout: <root>/notebooks/<this notebook>  and  <root>/data/...
def _find_project_root() -> str:
    cwd = os.getcwd()
    if os.path.basename(cwd) == "notebooks":
        return os.path.dirname(cwd)
    if os.path.isdir(os.path.join(cwd, "data")) or os.path.isdir(os.path.join(cwd, "notebooks")):
        return cwd
    p = cwd
    for _ in range(5):
        if os.path.isdir(os.path.join(p, "notebooks")) or os.path.isdir(os.path.join(p, "data")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            break
        p = parent
    return cwd

PROJECT_ROOT = _find_project_root()
DATA_ROOT    = os.path.join(PROJECT_ROOT, "data")
if not os.path.isdir(DATA_ROOT):
    raise RuntimeError(
        f"Expected a 'data' folder at {DATA_ROOT} (project root resolved to {PROJECT_ROOT}).\n"
        "Run this notebook from notebooks/ or the project root, or create data/ first."
    )

# ── Output paths ────────────────────────────────────────────────────────────
RAW_DATA_DIR  = os.path.join(DATA_ROOT, "0_raw", "results")
PROGRESS_FILE = os.path.join(DATA_ROOT, "0_raw", "progress.json")
ERROR_DIR     = os.path.join(DATA_ROOT, "0_raw", "errors")
LOG_FILE      = os.path.join(DATA_ROOT, "0_raw", "harvest.log")

# ── Logging ─────────────────────────────────────────────────────────────────
# Console + file. The file handler is critical for unattended multi-day runs:
# closing the notebook would otherwise lose the audit trail.
os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, encoding="utf-8"), logging.StreamHandler(sys.stdout)],
)
log = logging.getLogger("harvest")
log.info("project root: %s", PROJECT_ROOT)

# ── Date range ──────────────────────────────────────────────────────────────
START_YEAR,  END_YEAR  = 1994, 2025
START_MONTH, END_MONTH = 1, 12

# ── Query ───────────────────────────────────────────────────────────────────
# US-affiliated, English-language journal articles with abstracts (humans only).
# No disease pre-filter — disease detection happens downstream.
# Journal Article[pt] excludes editorials, letters, news, comments.
BASE_QUERY = (
    "english[Language]"
    " AND (USA[Affiliation] OR US[Affiliation])"
    " AND hasabstract[text]"
    " AND humans[Filter]"
    " AND Journal Article[pt]"
)

# ── Request settings ────────────────────────────────────────────────────────
# The 10,000 wall: for db=pubmed, retstart + retmax must be <= 10,000 on every
# efetch — including efetch-from-history. The history server does not lift this.
# Workaround: split any query whose count exceeds SAFE_SET_MAX into finer date
# ranges (month → days → half-days) so every piece pages in full.
EFETCH_OFFSET_CAP = 10_000      # NCBI hard limit
SAFE_SET_MAX      = 9_500       # split threshold, 500-record margin under the cap
HISTORY_BATCH     = 500         # records per efetch call
LOG_EVERY         = 10          # log progress every Nth batch
MAX_RETRIES       = 6           # HTTP retries per request
CHUNK_RETRIES     = 3           # parse-error retries per efetch batch
TIMEOUT           = (10, 180)   # (connect, read) seconds

# Stay under the NCBI ceiling (10 req/s with API key, 3 req/s without).
MIN_INTERVAL = 1.0 / 9.0 if API_KEY else 1.0 / 2.5

## 2. HTTP session and rate-limited request helper

One global throttle covers every E-utilities call (esearch and efetch), so bursts
never exceed the ceiling. The interval is **adaptive**: repeated HTTP 429s widen
it automatically — which protects you if an API key is silently invalid (NCBI
then enforces the 3 req/s IP limit while you think you have 10).

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": f"{TOOL}/2.0 (mailto:{EMAIL})"})

_state = {"last_call": 0.0, "interval": MIN_INTERVAL, "consec_429": 0}

def _throttle() -> None:
    wait = _state["interval"] - (time.time() - _state["last_call"])
    if wait > 0:
        time.sleep(wait)
    _state["last_call"] = time.time()

def _note_429() -> None:
    """Two+ consecutive 429s => we're over the real ceiling; back the global rate down."""
    _state["consec_429"] += 1
    if _state["consec_429"] >= 2 and _state["interval"] < 0.5:
        _state["interval"] = min(0.5, _state["interval"] * 1.5)
        log.warning("repeated 429s -> widening request interval to %.3fs "
                    "(is the API key valid? keyless limit is 3 req/s)", _state["interval"])

def _note_ok() -> None:
    _state["consec_429"] = 0

def _with_identity(params: dict) -> dict:
    p = dict(params)
    p["tool"] = TOOL
    p["email"] = EMAIL
    if API_KEY:
        p["api_key"] = API_KEY
    return p

def eutils_request(endpoint: str, params: dict, method: str = "GET") -> requests.Response:
    """Rate-limited E-utilities call. Retries on transient network / HTTP 429,5xx errors.
    Raises RuntimeError after MAX_RETRIES so the caller can decide what to do."""
    url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/{endpoint}"
    params = _with_identity(params)
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        _throttle()
        try:
            if method == "POST":
                resp = SESSION.post(url, data=params, timeout=TIMEOUT)
            else:
                resp = SESSION.get(url, params=params, timeout=TIMEOUT)
            if resp.status_code == 429:
                _note_429()
                raise requests.exceptions.HTTPError("429 Too Many Requests")
            if resp.status_code in (500, 502, 503, 504):
                raise requests.exceptions.HTTPError(f"{resp.status_code} {resp.reason}")
            resp.raise_for_status()
            _note_ok()
            return resp
        except requests.exceptions.RequestException as e:
            last_err = e
            backoff = min(60.0, 2.0 ** attempt) + random.uniform(0, 1)
            log.info("    request failed (%s); retry %d/%d in %.1fs", e, attempt, MAX_RETRIES, backoff)
            time.sleep(backoff)
    raise RuntimeError(f"{endpoint} failed after {MAX_RETRIES} retries: {last_err}")

## 3. Post a search to the history server

For `db=pubmed`, no result set can be paged past offset 10,000 — esearch returns
only the first 10k PMIDs, and efetch against a history set rejects
`retstart >= 10000` with HTTP 400. The history server lets efetch *reference* a
stored set, but does not lift this cap.

The fix is to split any query whose count exceeds `SAFE_SET_MAX` into finer date
ranges (month → individual days → half-days) so every piece we actually page is
well under 10k. This function just posts one (already-bounded) query and returns
its handles and count. `retmax=0` keeps the response tiny — we only need the
count and the WebEnv / query_key.

In [ ]:
def post_search_history(query: str) -> tuple:
    """esearch with usehistory=y. Returns (webenv, query_key, total_count)."""
    params = {
        "db": "pubmed",
        "term": query,
        "usehistory": "y",
        "retmax": 0,
        "retmode": "json",
    }
    data = eutils_request("esearch.fcgi", params).json()
    r = data.get("esearchresult", {})

    if "ERROR" in r or "error" in data:
        raise RuntimeError(f"esearch error: {r.get('ERROR') or data.get('error')}")

    total = int(r.get("count", 0))
    webenv, query_key = r.get("webenv"), r.get("querykey")

    # Without WebEnv and query_key, efetch cannot retrieve the result set at all.
    if total > 0 and (not webenv or not query_key):
        raise RuntimeError("esearch returned no WebEnv/query_key; cannot page the result set")

    return webenv, query_key, total

## 4. Fetch one batch from the history server

efetch against `WebEnv` / `query_key` with `retstart` / `retmax` pages the full
stored set — no 10k cap here. `retmode=xml` (no `rettype`) returns the full
record including MeSH qualifiers and `CoiStatement`. Each batch retries on
transient XML parse errors; an HTML body means the history set expired (rare
for per-month sets) and is treated as a failure so the month resumes.

In [ ]:
def fetch_history_batch(webenv: str, query_key: str, retstart: int,
                        retmax: int = HISTORY_BATCH) -> list:
    """Fetch + parse one [retstart, retstart+retmax) window from the history server."""
    params = {"db": "pubmed", "query_key": query_key, "WebEnv": webenv,
              "retstart": retstart, "retmax": retmax, "retmode": "xml"}
    for attempt in range(1, CHUNK_RETRIES + 1):
        resp = eutils_request("efetch.fcgi", params, method="GET")
        body = resp.content
        head = body[:256].lstrip().lower()
        if head.startswith(b"<!doctype") or head.startswith(b"<html"):
            raise RuntimeError("efetch returned an HTML page (history set expired?)")
        try:
            root = ET.fromstring(body)  # bytes -> honors XML-declared encoding
        except ET.ParseError as e:
            if attempt == CHUNK_RETRIES:
                raise RuntimeError(f"XML parse failed after {CHUNK_RETRIES} attempts @ {retstart}: {e}")
            log.info("    XML parse error @ %d (attempt %d/%d); refetching", retstart, attempt, CHUNK_RETRIES)
            time.sleep(2.0 * attempt)
            continue
        return [a for a in (parse_article(el) for el in root.findall(".//PubmedArticle")) if a]
    return []  # unreachable

## 5. Parse PubmedArticle XML into dicts

`itertext()` preserves text across inline markup, so titles and abstracts cannot
be truncated at `<i>` or `<sup>`. Publication dates are resolved from the three
forms PubMed uses (structured `<PubDate>`, free-text `<MedlineDate>`,
`<ArticleDate>` e-pub fallback) and tagged with a precision flag — so a
season-only date like "1998 Spring" cannot be silently treated as January. A
parse failure on one record is logged and skipped; the batch keeps going.

In [ ]:
_MONTHS = {"jan": "01", "feb": "02", "mar": "03", "apr": "04",
           "may": "05", "jun": "06", "jul": "07", "aug": "08",
           "sep": "09", "oct": "10", "nov": "11", "dec": "12"}

def _text(elem) -> str:
    """Full text of an element including inline-markup tails (<i>, <sup>, ...).
    Plain .text would stop at the first inline tag and drop everything after it —
    a real problem for titles like "TNF-α in <i>Mycobacterium</i> infection"."""
    return "".join(elem.itertext()).strip() if elem is not None else ""

def parse_article(elem) -> Optional[dict]:
    """Parse one PubmedArticle element. Returns None if PMID or title is missing
    so the caller can filter out unusable records without raising."""
    try:
        pmid_elem = elem.find(".//PMID")
        if pmid_elem is None or not (pmid_elem.text or "").strip():
            return None
        pubdate, pubdate_raw, precision = _extract_pubdate(elem)
        return {
            "uid":               pmid_elem.text.strip(),
            "title":             _text(elem.find(".//ArticleTitle")),
            "journal":           _text(elem.find(".//Journal/Title")),
            "pubdate":           pubdate,            # normalized YYYY-MM-DD (best effort)
            "pubdate_raw":       pubdate_raw,        # original string as PubMed gave it
            "pubdate_precision": precision,          # "full_date" | "year_month" | "year" | ""
            "abstract_sections": _extract_abstract(elem),
            "authors":           _extract_authors(elem),
            "mesh_terms":        _extract_mesh(elem),
            "keywords":          [_text(k) for k in elem.findall(".//Keyword") if _text(k)],
            "coi_statement":     _text(elem.find(".//CoiStatement")),
        }
    except Exception as e:
        # Single bad record must not poison the whole batch — log and skip.
        log.info("    parse error for PMID %s: %s", elem.findtext(".//PMID", "?"), e)
        return None

def _extract_pubdate(elem) -> tuple:
    """Return (normalized 'YYYY-MM-DD', raw string, precision).

    PubMed dates come in three forms, in priority order:
      1. <PubDate> with structured Year/Month/Day      -> full_date / year_month / year
      2. <PubDate> with free-text <MedlineDate>        -> "1998 Dec-1999 Jan", "1994 Spring"
      3. <ArticleDate> (electronic pub date)           -> used when print metadata is missing

    The `precision` flag is what protects downstream seasonality work: "1998 Spring"
    must not be silently collapsed to January. It's normalized to a date for sorting,
    but the flag records what we actually knew.
    """
    pd = elem.find(".//PubDate")
    if pd is not None:
        year = pd.findtext("Year", "").strip()
        if year:
            # Happy path: structured date elements present.
            raw_month = pd.findtext("Month", "").strip()
            raw_day = pd.findtext("Day", "").strip()
            # Month can be a 3-letter abbreviation (Jan, Feb) or a number — handle both.
            month = _MONTHS.get(raw_month.lower()[:3], raw_month if raw_month.isdigit() else "")
            raw = " ".join(x for x in (year, raw_month, raw_day) if x)
            if month and raw_day:
                return f"{year}-{month.zfill(2)}-{raw_day.zfill(2)}", raw, "full_date"
            if month:
                return f"{year}-{month.zfill(2)}-01", raw, "year_month"
            return f"{year}-01-01", raw, "year"

        # Legacy free-text date. Grab the year always, the month only if recognizable.
        med = pd.findtext("MedlineDate", "").strip()
        if med:
            m = re.match(r"(\d{4})\s*([A-Za-z]{3})?", med)
            if m:
                mon = _MONTHS.get((m.group(2) or "").lower(), "")
                if mon:
                    return f"{m.group(1)}-{mon}-01", med, "year_month"
                # Season-only ("1994 Spring") or year range — drop to year precision.
                return f"{m.group(1)}-01-01", med, "year"

    # ArticleDate is the e-pub date — used for online-first papers without print metadata.
    ad = elem.find(".//ArticleDate")
    if ad is not None and ad.findtext("Year", "").strip():
        y = ad.findtext("Year").strip()
        mo = ad.findtext("Month", "").strip()
        da = ad.findtext("Day", "").strip()
        raw = " ".join(x for x in (y, mo, da) if x)
        if mo and da:
            return f"{y}-{mo.zfill(2)}-{da.zfill(2)}", raw, "full_date"
        if mo:
            return f"{y}-{mo.zfill(2)}-01", raw, "year_month"
        return f"{y}-01-01", raw, "year"

    return "", "", ""

def _extract_abstract(elem) -> list:
    """Structured abstracts have multiple AbstractText elements, each labelled
    (Background / Methods / Results / Conclusions). Unstructured abstracts have
    a single unlabelled one. Either way, return a list — downstream code is uniform."""
    return [{"label":    t.get("Label", ""),
             "category": t.get("NlmCategory", ""),
             "text":     "".join(t.itertext()).strip()}
            for t in elem.findall(".//AbstractText")]

def _extract_authors(elem) -> list:
    """Author list with name, ORCID (when present), and all affiliations.
    A record may have a CollectiveName ("The XYZ Study Group") instead of a person —
    keep it as the author entry rather than dropping the record."""
    authors = []
    for a in elem.findall(".//Author"):
        last  = a.findtext("LastName", "").strip()
        first = a.findtext("ForeName", "").strip()
        orcid = ""
        for idf in a.findall(".//Identifier"):
            if idf.get("Source") == "ORCID":
                orcid = (idf.text or "").strip()
        name = f"{last} {first}".strip() or _text(a.find("CollectiveName"))
        authors.append({
            "name":         name,
            "initials":     a.findtext("Initials", "").strip(),
            "orcid":        orcid,
            # Modern records can list multiple affiliations per author (joint appointments).
            "affiliations": [_text(x) for x in a.findall(".//AffiliationInfo/Affiliation") if _text(x)],
        })
    return authors

def _extract_mesh(elem) -> list:
    """MeSH (Medical Subject Headings) terms with their qualifiers (subheadings).

    Structure: each MeshHeading has one DescriptorName (the main term, e.g. "Diabetes
    Mellitus, Type 2") and zero or more QualifierName children (e.g. "therapy",
    "diagnosis"). MajorTopicYN flags whether the term is a primary subject of the
    paper — used for relevance weighting downstream.
    """
    terms = []
    for h in elem.findall(".//MeshHeading"):
        d = h.find("DescriptorName")
        if d is None:
            continue
        terms.append({
            "descriptor":  _text(d),
            "major_topic": d.get("MajorTopicYN", "N") == "Y",
            "qualifiers":  [{"name": _text(q), "major_topic": q.get("MajorTopicYN", "N") == "Y"}
                            for q in h.findall("QualifierName")],
        })
    return terms

## 6. Storage, progress, and resume helpers

JSONL append (one record per line) is O(1) and append-safe. Progress is written
atomically (temp file + `os.replace`) so the progress file itself cannot be
corrupted by an interrupt.

In [ ]:
def append_jsonl(path: str, records: list) -> None:
    """Append records to a JSONL file. O(1) per call, durable on return."""
    os.makedirs(os.path.dirname(path), exist_ok=True)

    # If a previous write was interrupted, the file may end mid-record with no
    # trailing newline. Start the new batch on a fresh line so the torn fragment
    # cannot fuse with the first new record and produce one corrupt JSON line.
    needs_nl = os.path.exists(path) and os.path.getsize(path) > 0
    if needs_nl:
        with open(path, "rb") as f:
            f.seek(-1, os.SEEK_END)
            needs_nl = f.read(1) != b"\n"

    with open(path, "a", encoding="utf-8") as f:
        if needs_nl:
            f.write("\n")
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
        f.flush()
        # fsync forces the OS to flush its buffers to disk — without this, a
        # power loss seconds after f.close() can still lose the data.
        os.fsync(f.fileno())


def repair_jsonl(path: str) -> None:
    """Drop a trailing partial line left by an interrupted write.

    Algorithm: scan backwards from EOF in 64 KB chunks looking for the last
    newline, then truncate everything after it. Backwards + chunked means we
    never load the whole file into memory — important for month files that can
    grow to hundreds of MB.
    """
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return

    with open(path, "r+b") as f:
        f.seek(0, os.SEEK_END)
        end = f.tell()

        # If the file already ends in a newline, nothing to repair.
        f.seek(end - 1)
        if f.read(1) == b"\n":
            return

        # Walk backwards in 64 KB chunks until we find a newline.
        pos, last_nl = end - 1, -1
        while pos > 0 and last_nl < 0:
            start = max(0, pos - 65536)
            f.seek(start)
            buf = f.read(pos - start)
            nl = buf.rfind(b"\n")
            if nl >= 0:
                last_nl = start + nl
            pos = start

        # Truncate just after the last good newline (or to empty if no newline at all).
        f.truncate(last_nl + 1 if last_nl >= 0 else 0)


def load_saved_pmids(path: str) -> set:
    """PMIDs already written to a month's JSONL — used for resume and dedup.

    The JSONL file is the source of truth for "what's saved"; the progress file
    only tracks which pieces have finished. So on resume we re-derive the saved
    PMID set from the file itself, never trusting an external counter.
    """
    seen = set()
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    seen.add(json.loads(line)["uid"])
                except (json.JSONDecodeError, KeyError):
                    # A torn or malformed line — repair_jsonl handles trailing damage;
                    # anything else gets silently skipped here. We'd rather under-count
                    # saved PMIDs (causing a few harmless re-fetches) than over-count.
                    continue
    return seen


def load_progress(path: str) -> dict:
    """Load the harvest progress file.

    Schema:
      completed_months:        list of "YYYY-MM" strings that are fully done
      in_progress:             dict for the month currently being worked on,
                               listing finished date-range pieces (or None)
      total_time_seconds:      cumulative runtime across resumes
      total_batches_processed: cumulative efetch batches across resumes

    If the file is corrupt we start fresh — the JSONL files remain the source
    of truth for what's actually been saved, so we lose only the timing totals.
    """
    default = {
        "completed_months":        [],
        "in_progress":             None,
        "total_time_seconds":      0.0,
        "total_batches_processed": 0,
    }
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                loaded = json.load(f)
            default.update(loaded)
        except json.JSONDecodeError:
            print(f"  WARNING: {path} is corrupt; starting fresh "
                  f"(JSONL files remain the source of truth for what's saved)")
    return default


def write_json_atomic(path: str, obj) -> None:
    """Write JSON atomically: write to a temp file in the same directory,
    fsync it, then os.replace() it onto the target path. This guarantees the
    target either contains the old content or the new content — never a
    half-written file, even if the process is killed mid-write.

    Same-directory tempfile matters: os.replace is atomic only within a single
    filesystem, and a tempdir on a different mount would break that.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=os.path.dirname(path), suffix=".tmp")
    with os.fdopen(fd, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)  # atomic on POSIX and Windows

## 7. Main harvest loop
Each month is covered by one or more **sub-queries**, each guaranteed to return fewer than
`SAFE_SET_MAX` records so it can be paged in full (offset stays under the 10k wall). A month
over the cap is split by day; a single day over the cap (common in older years, where PubMed
dates many year-only records to Jan 1) is split by **PMID range** (`lo:hi[UID]`), bisecting
recursively until each piece is under the cap. Unlike a calendar day, a UID range can always
be subdivided — down to a single PMID — so nothing can get stuck above 10k. This is the same
approach EDirect uses internally. All sub-queries write into the same monthly JSONL, deduped
by PMID.

**Resume.** Progress records which pieces of the in-progress month are already `done`. On
restart, finished pieces are skipped and the rest re-run; PMID dedup against the on-disk file
makes re-running a partial piece safe. This works because every month in 1994–2025 is frozen.

In [ ]:
def _pdat(lo: str, hi: str = None) -> str:
    """Build a [PDAT] term: a single day (lo) or an inclusive day range (lo:hi)."""
    return f"{lo}[PDAT]" if hi is None or hi == lo else f"{lo}:{hi}[PDAT]"

def _split_by_uid(term: str, lo: int, hi: int, out: list) -> None:
    """Recursively bisect the PMID range [lo,hi] for `term` until each piece is <= SAFE_SET_MAX.
    A [UID] range can always be split (down to a single PMID), so this never gets stuck the way
    date ranges do — this is the same technique EDirect uses internally for >10k sets."""
    sub = f"{term} AND {lo}:{hi}[UID]"
    _, _, n = post_search_history(sub)
    if n == 0:
        return
    if n <= SAFE_SET_MAX or lo >= hi:
        out.append((f"uid_{lo}_{hi}", sub, n))
        return
    mid = (lo + hi) // 2
    _split_by_uid(term, lo, mid, out)
    _split_by_uid(term, mid + 1, hi, out)

# Max PMID is well above current values; PubMed PMIDs are < ~10^9. This upper bound only needs
# to exceed the largest existing PMID, and [UID] ranges past the max simply return 0.
_UID_MAX = 99_999_999

def _build_pieces(base_query: str, year: int, month: int) -> list:
    """Return (piece_id, term, count) pieces covering the month, each <= SAFE_SET_MAX.
    Tries the whole month, then splits oversized parts first by day, then by PMID range
    (so even a single calendar day with >10k records is handled)."""
    whole = f"{base_query} AND {year}/{month:02d}[PDAT]"
    _, _, total = post_search_history(whole)
    if total <= SAFE_SET_MAX:
        return [(f"{year}-{month:02d}", whole, total)]

    log.info("  %d-%02d has %s (> %s) — splitting by day", year, month, f"{total:,}", f"{SAFE_SET_MAX:,}")
    pieces, ndays = [], calendar.monthrange(year, month)[1]
    for day in range(1, ndays + 1):
        d = f"{year}/{month:02d}/{day:02d}"
        day_term = f"{base_query} AND {_pdat(d)}"
        _, _, n = post_search_history(day_term)
        if n == 0:
            continue
        if n <= SAFE_SET_MAX:
            pieces.append((d, day_term, n))
        else:
            # A single day over the cap (common for older years where PubMed dates many
            # year-only records to Jan 1). Date can't go finer than a day, so split by PMID.
            log.info("    %s has %s — splitting by PMID range", d, f"{n:,}")
            _split_by_uid(day_term, 1, _UID_MAX, pieces)
    return pieces

def _fetch_piece(term: str, out_file: str, saved: set) -> tuple:
    """Post one bounded sub-query and page its full set into out_file (deduped). Returns (count, ok)."""
    webenv, query_key, count = post_search_history(term)
    if count == 0:
        return 0, True
    for retstart in range(0, count, HISTORY_BATCH):
        records = fetch_history_batch(webenv, query_key, retstart)   # may raise -> caller handles
        fresh = [r for r in records if r["uid"] not in saved]
        for r in fresh:
            saved.add(r["uid"])
        append_jsonl(out_file, fresh)
    return count, True

def harvest(base_query: str, start_year: int, end_year: int,
            start_month: int = 1, end_month: int = 12) -> None:
    progress = load_progress(PROGRESS_FILE)
    completed = set(progress.get("completed_months", []))

    for year in range(start_year, end_year + 1):
        m_lo = start_month if year == start_year else 1
        m_hi = end_month if year == end_year else 12
        for month in range(m_lo, m_hi + 1):
            key = f"{year}-{month:02d}"
            if key in completed:
                continue

            log.info("%s\nProcessing %s ...", "=" * 60, key)
            out_file = f"{RAW_DATA_DIR}/results_{year}_{month:02d}.jsonl"

            try:
                pieces = _build_pieces(base_query, year, month)
            except Exception as e:
                log.warning("  planning failed for %s: %s — will retry next run", key, e)
                continue

            month_total = sum(n for _, _, n in pieces)
            if month_total == 0:
                os.makedirs(RAW_DATA_DIR, exist_ok=True)
                open(out_file, "a", encoding="utf-8").close()
                completed.add(key)
                _mark_complete(progress, completed, year, month, 0.0)
                log.info("  %s: 0 articles (empty file created)", key)
                continue

            repair_jsonl(out_file)
            saved = load_saved_pmids(out_file)

            # Resume: which pieces of THIS month are already done?
            ip = progress.get("in_progress") or {}
            done_pieces = set(ip.get("done_pieces", [])) if (ip.get("year") == year and ip.get("month") == month) else set()
            log.info("  %s articles across %d date-range piece(s); %s already saved",
                     f"{month_total:,}", len(pieces), f"{len(saved):,}")

            t0, ok = time.time(), True
            for pid, term, n in pieces:
                if pid in done_pieces:
                    continue
                try:
                    _fetch_piece(term, out_file, saved)
                except Exception as e:
                    err = f"{ERROR_DIR}/failed_{year}_{month:02d}_{pid.replace('/', '-')}.json"
                    write_json_atomic(err, {"piece": pid, "term": term, "error": str(e)})
                    log.error("  piece %s FAILED: %s — stopping month (resumes next run)", pid, e)
                    ok = False
                    break
                done_pieces.add(pid)
                progress["in_progress"] = {
                    "year": year, "month": month, "done_pieces": sorted(done_pieces),
                    "articles_saved": len(saved), "total_expected": month_total,
                }
                progress["total_batches_processed"] = progress.get("total_batches_processed", 0) + 1
                write_json_atomic(PROGRESS_FILE, progress)
                log.info("  [%5.1f%%] piece %s done — %s/%s saved",
                         len(done_pieces) / len(pieces) * 100, pid, f"{len(saved):,}", f"{month_total:,}")

            if ok:
                n_missing = month_total - len(saved)   # deleted/withheld (normal); count only
                if n_missing > 0:
                    rate = n_missing / month_total
                    mf = f"{ERROR_DIR}/missing_{year}_{month:02d}.json"
                    write_json_atomic(mf, {"month": key, "expected": month_total, "saved": len(saved),
                                           "missing_count": n_missing, "missing_rate": round(rate, 4)})
                    lvl = log.warning if rate > 0.02 else log.info
                    lvl("  %s: %s of %s PMIDs not returned (%.2f%%%s)",
                        key, f"{n_missing:,}", f"{month_total:,}", rate * 100,
                        " — ABNORMAL, investigate" if rate > 0.02 else " — deleted/withheld")
                completed.add(key)
                _mark_complete(progress, completed, year, month, time.time() - t0)
                _cleanup_error_files(year, month)
                log.info("  %s complete in %.0fs (%s saved)", key, time.time() - t0, f"{len(saved):,}")

    log.info("\nDone. %d months complete; %s pieces processed total.",
             len(completed), f"{progress.get('total_batches_processed', 0):,}")


def _mark_complete(progress, completed, year, month, secs) -> None:
    progress["completed_months"] = sorted(completed)
    progress["in_progress"] = None
    progress["total_time_seconds"] = progress.get("total_time_seconds", 0.0) + secs
    write_json_atomic(PROGRESS_FILE, progress)

def _cleanup_error_files(year, month) -> None:
    for fp in glob.glob(f"{ERROR_DIR}/failed_{year}_{month:02d}_*.json"):
        try:
            os.remove(fp)
        except OSError:
            pass

## 8. Run download
Safe to interrupt at any point — rerun this cell to resume. Finished date-range pieces of the
in-progress month are skipped; PMID dedup makes re-running a partial piece safe, so you never
lose or duplicate articles.

In [ ]:
harvest(BASE_QUERY, START_YEAR, END_YEAR, START_MONTH, END_MONTH)

## 9. Verify and summarize

Counts saved articles per month and across the run, reports the date-precision
distribution (`full_date` / `year_month` / `year`), and spot-checks the first
few files for duplicate PMIDs (should be zero — both within-batch and
across-batch dedup run during the harvest). Surfaces any months where the
deleted/withheld rate exceeded 2% — those are worth a look. Everything below
that threshold is routine PubMed churn and does not fail the run.

In [ ]:
import collections

files = sorted(glob.glob(f"{RAW_DATA_DIR}/results_*.jsonl"))
total, precision = 0, collections.Counter()
for fp in files:
    with open(fp, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            total += 1
            try:
                precision[json.loads(line).get("pubdate_precision", "")] += 1
            except json.JSONDecodeError:
                precision["unparseable_line"] += 1
print(f"{len(files)} monthly files, {total:,} articles total")
print("date precision:", dict(precision))

# Duplicate-PMID sanity check within the first few files
for fp in files[:3]:
    with open(fp, "r", encoding="utf-8") as f:
        uids = [json.loads(l)["uid"] for l in f if l.strip()]
    dups = [u for u, c in collections.Counter(uids).items() if c > 1]
    print(os.path.basename(fp), "->", f"{len(uids):,} rows,", f"{len(dups)} duplicate PMIDs")

# Cross-check saved counts vs PubMed's reported totals for a few months (optional spot check):
# any large gap beyond the deleted/withheld rate is worth investigating.
# Surface deleted/withheld counts recorded during the run; flag abnormal months.
miss = sorted(glob.glob(f"{ERROR_DIR}/missing_*.json"))
if miss:
    rows = [json.load(open(m)) for m in miss]
    tot_missing = sum(r["missing_count"] for r in rows)
    abnormal = [r["month"] for r in rows if r.get("missing_rate", 0) > 0.02]
    print(f"\n{tot_missing:,} PMIDs across {len(rows)} months not returned "
          f"(deleted/withheld — see errors/missing_*.json)")
    if abnormal:
        print(f"  ABNORMAL missing rate (>2%) in: {', '.join(abnormal)} — worth investigating")

prog = load_progress(PROGRESS_FILE)
print(f"\n{len(prog.get('completed_months', []))} months marked complete; "
      f"{prog.get('total_batches_processed', 0):,} batches; "
      f"{prog.get('total_time_seconds', 0)/3600:.1f}h cumulative.")

RERUN

In [ ]:
harvest(BASE_QUERY, START_YEAR, END_YEAR, START_MONTH, END_MONTH)

## Notes / caveats

- **Affiliation coverage is era-dependent.** `(USA OR US)[Affiliation]` misses
  "United States", "U.S.", bare state names, and empty affiliation fields
  (common pre-2000); PubMed also only began recording *all* authors'
  affiliations around 2014. Accepted limitation — document in the publication's
  methodology, do not widen the query.

- **MeSH terms are sparse pre-2000, keywords are sparse pre-2012** — affects
  notebooks 05, 06, and 07. Do not pre-filter on either; document as
  limitations.

- **1994 may be a partial year** — left in the download; inspect the per-file
  counts in section 9 and decide whether to flag or exclude it downstream.

- **Parser kept on `xml.etree`, not `lxml`.** This stage is network-bound
  (rate-limited to ~9 req/s), so XML-parse speed is not the bottleneck and
  `lxml` buys nothing measurable here, while its API differences
  (`XMLSyntaxError` vs `ParseError`, namespace handling) risk parse
  regressions. Use `lxml` in notebook 01, which re-parses the saved corpus in
  a tight CPU-bound loop — that is where the speedup lands.

- **`[PDAT]` bucketing.** Months are filed by Date of Publication; `pubdate`
  in the JSON prefers `PubDate`, then `MedlineDate`, then `ArticleDate`, so
  it can differ slightly from the `[PDAT]` bucket. Season-only MedlineDates
  ("1998 Spring") resolve to month 01.

- **Resume model.** `progress.json` holds `completed_months` plus an
  `in_progress` block; the JSONL files are the source of truth for which
  PMIDs are saved.